# Import libs

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [2]:
import os
import json
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import average_precision_score
import torch.nn.functional as F

# Configure model

In [3]:
os.chdir('/home/jovyan/dnalm/')

In [4]:
rmt_model_path = Path('/home/jovyan/dnalm/runs/annotation_bert_large_shawerma_continued_from_best_5_classes_no_intergenic_combined_8k_bpe_full_up_to_250k_exon_level_choosing_middle_lr_big_wd_UNET_segmented_UNET_repeater_3_step/bert_large_512_lastln_t2t_1000G_bs256_lr_1e-04_fp16/model_1750000/rmt_seglen_512_len8192_maxnsegm_10000_msz_5_bptt-1_lr1e-05_AdamW_constant_with_warmup_wd1e-04_p10000_bs_it500000/run_1/')

In [5]:
exp_config = json.load((rmt_model_path / 'config.json').open('r')) # it should be config.json that I sent you

In [6]:
for k in ['input_seq_len', 'model_cfg', 'model_cls', 'backbone_cls', 'input_size', 'num_mem_tokens', 'max_n_segments', 'tokenizer']:
    print(f'{k}: {exp_config[k]}')

input_seq_len: 8192
model_cfg: ./data/configs/L24-H1024-A16-V32k-preln-lastln.json
model_cls: src.gena_lm.modeling_rmt:RMTEncoderForLetterLevelTokenClassificationUNETsegmentedRepeater
backbone_cls: src.gena_lm.modeling_bert:BertForLetterLevelTokenClassification
input_size: 512
num_mem_tokens: 5
max_n_segments: 10000
tokenizer: ./data/tokenizers/t2t_1000h_multi_32k/


In [7]:
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizers/t2t_1000h_multi_32k/')

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from src.gena_lm.modeling_bert import BertForLetterLevelTokenClassification

In [9]:
model_cfg = AutoConfig.from_pretrained('./data/configs/L24-H1024-A16-V32k-preln-lastln-copy.json') # here it soulbe config for backbone model, don't change it, you can change only path to it
model_cfg.num_labels = 5
model_cfg.problem_type = 'multi_label_classification'
model_cls = BertForLetterLevelTokenClassification
model = model_cls(config=model_cfg)

In [10]:
rmt_config = {
            'num_mem_tokens': exp_config['num_mem_tokens'],
            'max_n_segments': exp_config['max_n_segments'],
            'input_size': exp_config['input_size'],
            'bptt_depth': -1,
            'sum_loss': True,
            'tokenizer': tokenizer
        }
from src.gena_lm.modeling_rmt import RMTEncoderForLetterLevelTokenClassificationUnetSegmentedEmbeddingOnly
rmt_cls = RMTEncoderForLetterLevelTokenClassificationUnetSegmentedEmbeddingOnly
model = rmt_cls(model, **rmt_config)

In [11]:
# load pre-trained weights
ckpt = torch.load(str('/home/jovyan/dnalm/runs/annotation_bert_large_shawerma_continued_from_best_5_classes_no_intergenic_combined_8k_bpe_full_up_to_250k_exon_level_choosing_middle_lr_big_wd_UNET_segmented_UNET_repeater_3_step/bert_large_512_lastln_t2t_1000G_bs256_lr_1e-04_fp16/model_1750000/rmt_seglen_512_len8192_maxnsegm_10000_msz_5_bptt-1_lr1e-05_AdamW_constant_with_warmup_wd1e-04_p10000_bs_it500000/run_1/model_best/pytorch_model.bin'), map_location='cpu')
missing_k, unexpected_k = model.load_state_dict(ckpt, strict=False)
print(f'missing: {missing_k}') # if no missing tensors - that is correct, otherwise - no!
print(f'unexpected_k: {unexpected_k}') # if no missing tensors - that is correct, otherwise - no!

missing: []
unexpected_k: []


In [12]:
model = model.eval()
# model.half()

# Evaluation example

In [13]:
seq = 'ATGC'*1234
input_features = tokenizer(seq, return_tensors='pt')

input_features['labels_mask'] = input_features['attention_mask'] # dumb realization

In [14]:
input_features['input_ids'].shape

torch.Size([1, 619])

In [15]:
input_features.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels_mask'])

In [16]:
input_features['labels'] = torch.ones((input_features['input_ids'].shape[1], 5)).unsqueeze(axis=0) # yeah, for now you must specify whatever labels, model won't work without them, it does not change the prediction
# input_features['labels'] = None

In [17]:
input_features['labels'].shape

torch.Size([1, 619, 5])

In [ ]:
input_features['labels'].T[None, :, :]

In [18]:
input_features['labels_mask'].shape

torch.Size([1, 619])

In [18]:
# with torch.autocast(device_type='cuda', dtype=torch.float16):
with torch.no_grad():
    out = model(**input_features, output_hidden_states=True)

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/transformers/modeling_utils.py:1101: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [19]:
out.keys()

dict_keys(['all_memory_embeddings', 'labels_segm', 'rmt_logits_masks_segm'])

In [21]:
out['all_memory_embeddings'][1]

tensor([[[ 0.1612, -0.3453, -0.0903,  ..., -0.7527,  0.7983,  0.5882],
         [ 0.1987, -0.3132, -0.0816,  ..., -0.7530,  0.7953,  0.5135],
         [ 0.1815, -0.3306, -0.0857,  ..., -0.7753,  0.8121,  0.4802],
         [ 0.1868, -0.3838, -0.1109,  ..., -0.7682,  0.8198,  0.5138],
         [ 0.1898, -0.4034, -0.1091,  ..., -0.7697,  0.8323,  0.5414]]])

In [22]:
out['all_memory_embeddings'].shape

torch.Size([2, 1, 5, 1024])

In [24]:
out['labels_segm'].shape

torch.Size([2, 1, 512, 5])

In [25]:
out['rmt_logits_masks_segm'].shape

torch.Size([2, 1, 512])